# FIRMS Data collection (NASA FIRMS API) 
For our analysis, we use wildfire data provided by NASA’s LANCE program (Land, Atmosphere Near real-time Capability for Earth observation). The NASA FIRMS dataset provides access to MODIS data onboard the Aqua and Terra satellites, as well as Visible Infrared Imaging Radiometer Suite (VIIRS) data aboard the S-NPP, NOAA-20, and NOAA-21 satellites. In this analysis, only VIIRS data from all three available satellites is used due to its significantly higher spatial resolution (~375 m).

The core idea of this project is to create a script that analyzes the situation continuously based on the most recent available data. Therefore, an API is used to download up-to-date datasets from the mentioned sensors.

# Important: Time window can be adjusted!
Before running the script, the user can specify how many days back (starting from today) should be included in the wildfire analysis for South America. The default value is 1, corresponding to the last 24 hours. Only integer values between 1 and 7 are valid.

In [3]:
# Set analysis time window
time_window = 2

# Be aware of the API rate limits when setting the time window. 
# A larger time window may result in more data points, which could lead to hitting the API rate limits.
# Consider as well to adjust the other two parameters (MAX_DISTANCE_KM and TIME_THRESHOLD) in the Analysis.ipynb notebook
# to better suit the chosen time window and the expected density of fire detections.

In [4]:
import requests
import pandas as pd
import os
from io import StringIO
from datetime import datetime, timezone

# --------------------------------------------------
# NASA FIRMS API Settings
# --------------------------------------------------

MAP_KEY = "01b9b20f1c9e80d44560acd21f5a79a7"

# Bounding Box for South America
# west,south,east,north
AREA = "-85,-57,-32,14"

# --------------------------------------------------
# Sensors 
# --------------------------------------------------

SENSORS = [
    "VIIRS_NOAA21_NRT",
    "VIIRS_NOAA20_NRT",
    "VIIRS_SNPP_NRT"
]

#day range in days (1 = last 24 hours, 7 = last 7 days, etc.)
DAY_RANGE = time_window

# --------------------------------------------------
# Output folder
# --------------------------------------------------

OUTPUT_DIR = os.path.join("..", "data", "raw")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting download for all sensors...")

for SOURCE in SENSORS:

    print(f"Loading data for sensor: {SOURCE}")

    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{MAP_KEY}/{SOURCE}/{AREA}/{DAY_RANGE}"
    )

    print(url)

    try:
        response = requests.get(url)
        response.raise_for_status()

        df = pd.read_csv(StringIO(response.text))

        print(f"Data points found ({SOURCE}): {len(df)}")

        timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

        filename = f"firms_{SOURCE}_south_america_{timestamp}.csv"
        file_path = os.path.join(OUTPUT_DIR, filename)

        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Data saved: {file_path}")
        else:
            print("No data found for this sensor.")

    except Exception as e:
        print(f"Error with {SOURCE}: {e}")

print("Finished.")


Starting download for all sensors...
Loading data for sensor: VIIRS_NOAA21_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_NOAA21_NRT/-85,-57,-32,14/2
Data points found (VIIRS_NOAA21_NRT): 3046
Data saved: ..\data\raw\firms_VIIRS_NOAA21_NRT_south_america_20260529_102832.csv
Loading data for sensor: VIIRS_NOAA20_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_NOAA20_NRT/-85,-57,-32,14/2
Data points found (VIIRS_NOAA20_NRT): 3063
Data saved: ..\data\raw\firms_VIIRS_NOAA20_NRT_south_america_20260529_102834.csv
Loading data for sensor: VIIRS_SNPP_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_SNPP_NRT/-85,-57,-32,14/2
Data points found (VIIRS_SNPP_NRT): 3399
Data saved: ..\data\raw\firms_VIIRS_SNPP_NRT_south_america_20260529_102836.csv
Finished.
